# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [11]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [12]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [13]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [14]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [15]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [16]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [17]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [18]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [19]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [20]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times among the projects listed.'

In [21]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there were use cases related to security. Specifically, one project titled "MediMind" is focused on security, involving a medical imaging solution aimed at improving early diagnosis through vision transformers.'

In [22]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had various comments about the fintech-related projects. Overall, they found many of these projects to be promising and impressive:\n\n- One project was described as "A clever solution with measurable environmental benefit."\n- Another was praised for being "Technically ambitious and well-executed."\n- A different project was considered a "Promising idea with robust experimental validation."\n- Some projects received positive remarks such as "Solid work with impressive real-world impact."\n- One project was noted for having "Excellent code quality and use of open-source libraries."\n- Another was characterized as "Strong quantitative results; add qualitative analysis next time."\n- The final project mentioned received comments that it had "Minor issues with integration but otherwise very polished."\n\nIn summary, judges generally viewed these fintech projects as innovative, well-developed, and impactful, with some noting minor areas for improvement.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [23]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [24]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [25]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the most common project domain is not explicitly specified, but among the examples given, the domains listed include Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. Since I only have a few samples and no data showing the overall frequency, I cannot definitively determine the most common domain.\n\nHowever, if these samples are representative, there doesn't seem to be a clear majority domain. Could you provide more data or specify how many projects belong to each domain?"

In [26]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided information, there are no explicit mentions of use cases related to security.'

In [27]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech project "PulseAI 50" as technically ambitious and well-executed.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer
In cases where I need a perfect keyword search looking for exact matches. The example I can think of is error codes, quotes, in cases like CRM apps order retrieval by order number, tracking shipping with a AWB etc. Instances where we want to match the exact charset in the question

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [28]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [29]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [30]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, it appears that the project domains include Healthcare / MedTech, Creative / Design / Media, and Security. Since only a few examples are given, I cannot definitively determine the most common project domain. If the full dataset were available, sorting by frequency would help identify the most common domain. \n\nHowever, from the sample provided, Healthcare / MedTech appears twice, suggesting it might be a common domain in this dataset.'

In [31]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security explicitly mentioned. The projects described focus on federated learning to improve privacy in healthcare applications, which is related to security and privacy, but there are no direct mentions of security use cases.'

In [32]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. Specifically, for the project "PlanPilot 35" in the Finance/FinTech domain, the judges described it as "A clever solution with measurable environmental benefit."'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [33]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [34]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [35]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Legal / Compliance," as it is mentioned multiple times across different projects.'

In [36]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. Specifically, the project titled "InsightAI 1" with the project name "Project Aurora" falls under the security domain. Its description is "A low-latency inference system for multimodal agents in autonomous systems," indicating a focus on security applications in autonomous systems.'

In [37]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had various comments about the fintech projects. For example, one judge noted that the project was "Conceptually strong but results need more benchmarking," indicating a positive view of the concept but highlighting the need for more validation. Another project was described as having a "Solid work with impressive real-world impact," showing strong approval for its practical applications. Overall, the judges appreciated the innovative ideas, technical quality, and potential for real-world impact in the fintech projects, though some suggestions for further benchmarking and validation were mentioned.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer
Broadening the search to capture different semantic interpretations and keywords associated with the initial request. A single, poorly-phrased query might fail to retrieve relevant documents, but a set of diverse, expanded queries increases the chances of a successful match.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [38]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [39]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [40]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [41]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [42]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [43]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times among the sample projects.'

In [44]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly related to security. The projects mentioned focus on federated learning and improving privacy in healthcare applications, which can be related to security concerns, but there are no direct references to security use cases.'

In [45]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive feedback about the fintech projects. Specifically, one project described as "technically ambitious and well-executed," indicating a high regard for its quality. Another project was noted for being "promising" with "robust experimental validation," showing confidence in its potential. Additionally, a project was praised for its "comprehensive and technically mature approach." Overall, the judges recognized the projects for their promising ideas, solid work, and impactful contributions.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [46]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [47]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [48]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times among the projects.'

In [49]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security in the provided context. Specifically, the project titled "MediMind 17" falls under the domain of Security. Its description is: "A medical imaging solution improving early diagnosis through vision transformers," which indicates a focus on security in healthcare imaging.'

In [50]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a positive view of the fintech projects. For example, they described the "Pathfinder 27" project as having excellent code quality and the use of open-source libraries. Overall, the judges appreciated the quality, scalability, and real-world impact of these projects, though some noted minor issues with integration.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [51]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [52]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [53]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [54]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [55]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [56]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Legal / Compliance," which is mentioned twice. Other domains like "Developer Tools / DevEx," "Productivity Assistants," and "Customer Support / Helpdesk" are also present, but less frequently. Therefore, based on this data, "Legal / Compliance" is the most common project domain.'

In [57]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security mentioned in the provided context. Specifically, one project titled "SynthMind" is a medical imaging solution that involves improving early diagnosis through vision transformers, and it is categorized under the secondary domain "Security." Additionally, another project titled "Project Aurora" describes a low-latency inference system for multimodal agents in autonomous systems, which can also be related to security in autonomous or security-critical applications.'

In [58]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had the following comments about the fintech projects:\n\n1. **TrendLens 19**: Judges said it was "Technically ambitious and well-executed."\n2. **WealthifyAI 16**: Described as "Comprehensive and technically mature approach."\n3. **AutoMate 5**: Noted as "A forward-looking idea with solid supporting data."\n4. **LearnWise 15**: Mentioned as "Solid work with impressive real-world impact."\n5. **InsightAI 1**: Commented as "Technically ambitious and well-executed."\n\nOverall, the judges recognized these fintech projects as ambitious, well-executed, and impactful, with several highlighting their technical maturity and forward-looking nature.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
I believe it will leave out the nuances and start giving out generic/similar answers to questions that "seem" semantically identical. One option I can think of is to use gradient alongside percentile. Per the documentation, The idea is to apply anomaly detection on gradient array so that the distribution become wider and easy to identify boundaries in highly semantic data.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [59]:
### YOUR CODE HERE

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [60]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [61]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg


KnowledgeGraph(nodes: 0, relationships: 0)

In [62]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

In [63]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'f93082'. Skipping!
Property 'summary' already exists in node 'b85dc9'. Skipping!
Property 'summary' already exists in node '974bd8'. Skipping!
Property 'summary' already exists in node '8e0d27'. Skipping!
Property 'summary' already exists in node '22a25a'. Skipping!
Property 'summary' already exists in node '4b582e'. Skipping!
Property 'summary' already exists in node '4cc917'. Skipping!
Property 'summary' already exists in node '0e16ef'. Skipping!
Property 'summary' already exists in node '9546ee'. Skipping!
Property 'summary' already exists in node '2f4dec'. Skipping!
Property 'summary' already exists in node 'cb6b65'. Skipping!
Property 'summary' already exists in node 'cad652'. Skipping!
Property 'summary' already exists in node '2cfe00'. Skipping!
Property 'summary' already exists in node '28d970'. Skipping!
Property 'summary' already exists in node 'c86b92'. Skipping!
Property 'summary' already exists in node '678aa1'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'b85dc9'. Skipping!
Property 'summary_embedding' already exists in node '8e0d27'. Skipping!
Property 'summary_embedding' already exists in node 'f93082'. Skipping!
Property 'summary_embedding' already exists in node '4cc917'. Skipping!
Property 'summary_embedding' already exists in node '974bd8'. Skipping!
Property 'summary_embedding' already exists in node 'cad652'. Skipping!
Property 'summary_embedding' already exists in node 'cb6b65'. Skipping!
Property 'summary_embedding' already exists in node '9546ee'. Skipping!
Property 'summary_embedding' already exists in node '0e16ef'. Skipping!
Property 'summary_embedding' already exists in node '22a25a'. Skipping!
Property 'summary_embedding' already exists in node '4b582e'. Skipping!
Property 'summary_embedding' already exists in node '2f4dec'. Skipping!
Property 'summary_embedding' already exists in node 'c86b92'. Skipping!
Property 'summary_embedding' already exists in node '28d970'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

In [64]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 711)

In [65]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

In [66]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [67]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What does Eloundou refer to in AI research?,[Introduction ChatGPT launched in November 202...,The provided context does not specify what Elo...,single_hop_specifc_query_synthesizer
1,OpenAI how does it help work?,[Table 1: ChatGPT daily message counts (millio...,The context explains that OpenAI's ChatGPT is ...,single_hop_specifc_query_synthesizer
2,What is the Secton in the context of ChatGPT u...,[Variation by Occupation Figure 23 presents va...,The context does not provide a definition or e...,single_hop_specifc_query_synthesizer
3,How does computer programming relate to the us...,[Conclusion This paper studies the rapid growt...,"According to the context, computer programming...",single_hop_specifc_query_synthesizer
4,How does the rapid growth of ChatGPT and its w...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth of ChatGPT, launched in Novem...",multi_hop_abstract_query_synthesizer
5,so like how does ChatGPT use vary by job like ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context shows that ChatGPT usage varies si...,multi_hop_abstract_query_synthesizer
6,How does the rapid growth in ChatGPT message v...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The rapid increase in ChatGPT message volume, ...",multi_hop_abstract_query_synthesizer
7,US ChatGPT use mostly in work or non-work and ...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context shows that in the US, as of July 2...",multi_hop_specific_query_synthesizer
8,Based on the growth of ChatGPT usage in the US...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context indicates that as of July 2025, ov...",multi_hop_specific_query_synthesizer
9,wHAT hAPPENED iN jULY 2025 wITH cHatGPT uSAGE ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"In July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer


Now Lets create Langsmith dataset

In [68]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
     # optional (defaults to this)
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [69]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

In [71]:



from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE10"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

In [73]:
for data_row in testset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [74]:
rag_documents = docs

In [75]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

In [76]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [77]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [78]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [79]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

In [80]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

In [81]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [82]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways including performing workplace tasks to augment or automate human labor, producing writing, software code, spreadsheets, and other digital products. Users interact with AI by asking questions, getting advice, and generating outputs, functioning either as co-workers or co-pilots that help improve productivity and problem-solving. Additionally, AI use spans both work-related activities and self-expression tasks, with users seeking information, advice, and creative assistance beyond traditional web search engines.'

In [83]:
eval_llm = ChatOpenAI(model="gpt-4.1")

In [84]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

In [85]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'respectful-company-99' at:
https://smith.langchain.com/o/57b6b3f2-a3af-4e79-a658-01991901c637/datasets/e4eea595-164d-499f-94cd-82eb0d37e420/compare?selectedSessions=2d210d6a-f2f7-4465-b780-6781fac722f1




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,what happened in july 2025 with chatgpt and ho...,"In July 2025, ChatGPT had more than 700 millio...",None,"In july 2025, chatgpt was used weekly by more ...",1,1,0,2.027393,354a867a-5445-4d2f-922f-343ef0b7c12a,d33e60f9-a5eb-4ebc-8b6b-bf72a15811e9
1,wHAT hAPPENED iN jULY 2025 wITH cHatGPT uSAGE ...,"In July 2025, ChatGPT had more than 700 millio...",None,"In July 2025, ChatGPT had been used weekly by ...",1,1,0,2.859572,1e9a8d96-69a3-4afb-8c4c-dbd069e5f4ca,e69a0fc3-c348-45f4-935f-8fc2c5362888
2,Based on the growth of ChatGPT usage in the US...,"Based on the context, the increasing trend of ...",None,"The context indicates that as of July 2025, ov...",1,1,0,4.229974,bcd841c6-e958-4516-9004-1d0adb1d187a,5e9b69c3-f47d-43f9-a840-9b2fdc9fdd6f
3,US ChatGPT use mostly in work or non-work and ...,"Based on the provided context, in the US, Chat...",None,"The context shows that in the US, as of July 2...",1,1,0,5.767193,8a5c278a-5ebd-409b-b7a1-ddf2e918ff95,b52a82d3-7b22-4757-95d6-89972911475a
4,How does the rapid growth in ChatGPT message v...,"The rapid growth in ChatGPT message volume, pa...",None,"The rapid increase in ChatGPT message volume, ...",1,1,0,3.342355,e3d204c0-226c-4eca-89f5-c690d6578670,60a9caf2-39f5-4300-8667-afb36b8ee1ec
5,so like how does ChatGPT use vary by job like ...,ChatGPT usage varies by occupation in terms of...,None,The context shows that ChatGPT usage varies si...,1,1,0,4.860871,eb7c451d-c5dc-4080-823a-03274257a1f1,91a17931-1818-4ed6-922a-57f7a955f247
6,How does the rapid growth of ChatGPT and its w...,The rapid growth of ChatGPT and its widespread...,None,"The rapid growth of ChatGPT, launched in Novem...",1,1,0,3.408842,5f5d62e5-5df5-44d3-b32d-f73b5681936f,2e963a83-9358-4a2a-9e6a-7beec3278736
7,How does computer programming relate to the us...,"Based on the context, computer programming acc...",None,"According to the context, computer programming...",1,1,0,2.367287,5e57aba0-8b8f-4bf5-b519-bbe5afb11a14,31dbdb70-e202-4175-9ed4-d64db2faf7ff
8,What is the Secton in the context of ChatGPT u...,The Section in the context of ChatGPT usage da...,None,The context does not provide a definition or e...,0,0,0,3.582979,438b859e-ad07-46ff-ba38-2810f410f31d,b3d62287-9b4b-40c7-b807-8c670962f8c7
9,OpenAI how does it help work?,"Based on the provided context, OpenAI helps wo...",None,The context explains that OpenAI's ChatGPT is ...,1,1,0,2.622122,49a8ce39-29c5-48e1-b591-6f864845d2d5,e6fe342e-50d8-4661-ac5b-1501797c7b45


Changing Retriever

BM 25 Reriever

In [86]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(testset)
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

AttributeError: 'TestsetSample' object has no attribute 'page_content'

In [ ]:
evaluate(
    bm25_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)